In [5]:
import requests
url = ("https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_110m_populated_places.geojson")
def fetch_geojson(url):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raises an exception for HTTP errors
        geojson_data = response.json()  # Parse the JSON response
        return geojson_data
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Error connecting to the server: {conn_err}")
    except Exception as err:
        print(f"An error occurred: {err}")
    return None

In [9]:
geojson_data = fetch_geojson(url)

if geojson_data:
    features = geojson_data.get("features", [])
    print(f"Số thành phố: {len(features)}")
    # Extract city ids and their coordinates
    for feature in features[:5]:  # Display first 5 cities
        city_name = feature["properties"].get("NAME")
        country_name = feature["properties"].get("ADM0NAME")
        coordinates = feature["geometry"]["coordinates"]
        print(f"Tên: {city_name}, Quốc gia: {country_name}, Tọa độ: {coordinates}")

Số thành phố: 243
Tên: Vatican City, Quốc gia: Vatican (Holy Sea), Tọa độ: [12.453386544971766, 41.903282179960115]
Tên: San Marino, Quốc gia: San Marino, Tọa độ: [12.441770157800141, 43.936095834768004]
Tên: Vaduz, Quốc gia: Liechtenstein, Tọa độ: [9.516669472907267, 47.13372377429357]
Tên: Lobamba, Quốc gia: Swaziland, Tọa độ: [31.19999710971274, -26.466667461352472]
Tên: Luxembourg, Quốc gia: Luxembourg, Tọa độ: [6.130002806227083, 49.611660379121076]


In [ ]:
content = """Hanoi,21.0285,105.8542
DaNang,ChuKhongPhaiSo,108.2022
DongBiThieuDuLieu
TPHCM,10.8231,106.6297
SaoHoa,200.5,150.1
CanTho,10.0452,105.7469"""

with open("test.txt", "w", encoding="utf-8") as f:
    f.write(content)

In [20]:
import os
# --- HÀM 1: ĐỌC FILE (Xử lý đầy đủ ngoại lệ) ---
def read_cities(filename):
    data = []
    
    # 1. Kiểm tra file có tồn tại không trước khi mở
    if not os.path.exists(filename):
        print(f"Lỗi: File '{filename}' không tồn tại.")
        return []

    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue 

                try:
                    parts = line.split(',')
                    
                    # 2. Kiểm tra định dạng (đủ cột không)
                    if len(parts) < 3:
                        raise ValueError("Thiếu dữ liệu")
                    
                    name = parts[0].strip()
                    
                    # 3. Kiểm tra kiểu dữ liệu (có phải số không)
                    try:
                        lat = float(parts[1])
                        lon = float(parts[2])
                    except ValueError:
                        raise ValueError("Tọa độ không phải là số")

                    # 4. Kiểm tra logic (Tọa độ có nằm trong phạm vi Trái đất không)
                    if not (-90 <= lat <= 90) or not (-180 <= lon <= 180):
                        raise ValueError(f"Tọa độ vô lý: ({lat}, {lon})")

                    data.append({"name": name, "lat": lat, "lon": lon})

                except ValueError as err:
                    # Bắt lỗi từng dòng để không dừng chương trình
                    print(f"Bỏ qua dòng lỗi '{line}': {err}")

    except UnicodeDecodeError:
        print(f"Lỗi: File sai định dạng mã hóa (encoding).")
    except Exception as err:
        print(f"Lỗi không xác định khi đọc file: {err}")

    return data




In [21]:

# --- HÀM 2: GHI FILE (Xử lý đầy đủ ngoại lệ) ---
def write_cities(filename, data):
    if not data:
        print("Không có dữ liệu để ghi.")
        return False

    try:
        with open(filename, 'w', encoding='utf-8') as f:
            for city in data:
                line = f"City: {city['name']} | Coords: ({city['lat']}, {city['lon']})\n"
                f.write(line)
        
        print(f"Đã ghi thành công vào file '{filename}'.")
        return True

    except PermissionError:
        print(f"Lỗi: Không có quyền ghi vào file '{filename}' (File đang mở?).")
    except FileNotFoundError:
        print(f"Lỗi: Đường dẫn thư mục không tồn tại.")
    except IOError as err:
        print(f"Lỗi hệ thống khi ghi file: {err}")
    
    return False

In [22]:

# --- HÀM 3: XỬ LÝ CHÍNH ---
def process_data(input_file, output_file):
    # Đọc (đã bao gồm xử lý lỗi)
    raw_data = read_cities(input_file)
    
    # Xử lý (Lọc)
    valid_data = [city for city in raw_data if city['lat'] > 10]
    
    print(f"Lọc được {len(valid_data)} dòng hợp lệ.")

    # Ghi (đã bao gồm xử lý lỗi)
    write_cities(output_file, valid_data)




In [23]:
process_data("test.txt", "result.txt")

print("\n--- Test file không tồn tại ---")
read_cities("file_ao.txt")

Bỏ qua dòng lỗi 'DaNang,ChuKhongPhaiSo,108.2022': Tọa độ không phải là số
Bỏ qua dòng lỗi 'DongBiThieuDuLieu': Thiếu dữ liệu
Bỏ qua dòng lỗi 'SaoHoa,200.5,150.1': Tọa độ vô lý: (200.5, 150.1)
Lọc được 3 dòng hợp lệ.
Đã ghi thành công vào file 'result.txt'.

--- Test file không tồn tại ---
Lỗi: File 'file_ao.txt' không tồn tại.


[]